# EX_09 — Agentes con LangChain (ejercicios)

**Notebook de referencia:** `notebook/09_Agentes_LangChain.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Tool decorator

Define un `@tool` (LangChain) que calcule el número de palabras de un texto. Prueba `.invoke` con un string.

*Hint:* `from langchain_core.tools import tool`.


In [1]:
from langchain_core.tools import tool

@tool
def count_words(text: str) -> int:
    """Cuenta el número de palabras en un texto (separadas por espacios)."""
    return len(text.split())

# Prueba directa de la tool (sin LLM)
sample = "Te damos la bienvenida al asistente virtual de EcoMarket"
result = count_words.invoke({"text": sample})
print(f"Texto: {sample!r}")
print(f"Palabras: {result}")


Texto: 'Te damos la bienvenida al asistente virtual de EcoMarket'
Palabras: 9


## Actividad 2 — Agente mínimo

Monta un agente (o `create_react_agent` según la versión de tu curso) con **un** LLM y la tool anterior. Si no tienes credenciales, deja el código comentado pero completo.


In [2]:
import os
from getpass import getpass
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd() / ".env")
    load_dotenv(Path.cwd().parent / ".env")
except ImportError:
    pass

_PLACEHOLDER_KEYS = {"", "gsk_reemplaza_con_tu_clave", "your-api-key-here", "changeme"}


def ensure_groq_api_key(*, reprompt: bool = False) -> str:
    """Obtiene una GROQ_API_KEY válida; repregunta si falta o es placeholder."""
    key = "" if reprompt else os.environ.get("GROQ_API_KEY", "").strip()
    if not key or key in _PLACEHOLDER_KEYS or not key.startswith("gsk_"):
        key = getpass("Introduce tu GROQ API Key (https://console.groq.com): ").strip()
        if not key or not key.startswith("gsk_"):
            raise ValueError(
                "Clave inválida. Crea una en https://console.groq.com (formato gsk_...)."
            )
        os.environ["GROQ_API_KEY"] = key
    return key


def setup_llm():
    from langchain_groq import ChatGroq

    key = ensure_groq_api_key()
    llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0, groq_api_key=key)
    try:
        llm.invoke("Responde solo: OK")
    except Exception as exc:
        err = str(exc).lower()
        if "401" in err or "invalid_api_key" in err:
            os.environ.pop("GROQ_API_KEY", None)
            key = ensure_groq_api_key(reprompt=True)
            llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0, groq_api_key=key)
            llm.invoke("Responde solo: OK")
        else:
            raise
    return llm


llm = setup_llm()
print("LLM configurado ✓")

LLM configurado ✓


In [3]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente. Usa count_words cuando te pidan contar palabras."),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, [count_words], prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=[count_words],
    verbose=True,
    max_iterations=5,
)

response = agent_executor.invoke({
    "input": "¿Cuántas palabras hay en esta frase?: Hola, soy tu asistente virtual",
    "chat_history": [],
})
print("\nRespuesta final:", response["output"])



> Entering new AgentExecutor chain...

Invoking: `count_words` with `{'text': 'Hola, soy tu asistente virtual'}`


5La frase "Hola, soy tu asistente virtual" contiene 5 palabras.

> Finished chain.

Respuesta final: La frase "Hola, soy tu asistente virtual" contiene 5 palabras.


## Actividad 3 — Traza deseada

Escribe la secuencia ideal de eventos (Thought / Action / Observation) para la pregunta: "How many words in this sentence: I love RAG?"


**Secuencia ideal (patrón ReAct) para:** *"How many words in this sentence: I love RAG?"*

1. **Thought:** El usuario pide contar palabras en una frase concreta; debo usar la herramienta `count_words` con el texto exacto.
2. **Action:** `count_words(text="I love RAG?")`
3. **Observation:** `3`
4. **Thought:** Ya tengo el resultado numérico; puedo responder al usuario.
5. **Final Answer:** *"The sentence 'I love RAG?' contains 3 words."*
